# A vs B/C Final Model

In [1]:
# Import necessary libraries
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import tensorflow as tf

from tensorflow.keras import layers, models, callbacks, regularizers

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    roc_curve
)

from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Reproducibility
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

## Load Tensors

In [3]:
#Load tensors
TENSOR_DIR = Path("/Users/omarleosamman/Downloads/pfizer_tensors")

X_torch = torch.load(TENSOR_DIR / "X_features.pt")
y_torch = torch.load(TENSOR_DIR / "y_labels.pt")
fold_torch = torch.load(TENSOR_DIR / "folds.pt")

X_tf = tf.convert_to_tensor(X_torch.cpu().numpy(), dtype=tf.float32)
y_tf = tf.convert_to_tensor(y_torch.cpu().numpy(), dtype=tf.int64)
fold_tf = tf.convert_to_tensor(fold_torch.cpu().numpy(), dtype=tf.int64)

print("X:", X_tf.shape)
print("y:", y_tf.shape)
print("folds:", fold_tf.shape)

X: (20931, 86, 65)
y: (20931,)
folds: (20931,)


In [4]:
# Keep only labeled data
labeled_mask = y_tf != -1

X_labeled = tf.boolean_mask(X_tf, labeled_mask)
y_labeled = tf.boolean_mask(y_tf, labeled_mask)
fold_labeled = tf.boolean_mask(fold_tf, labeled_mask)

print("X labeled:", X_labeled.shape)
print("y labeled:", y_labeled.shape)
print("fold labeled:", fold_labeled.shape)

print(pd.Series(y_labeled.numpy()).value_counts().sort_index())

X labeled: (11899, 86, 65)
y labeled: (11899,)
fold labeled: (11899,)
0    6406
1    3349
2    2144
Name: count, dtype: int64


In [5]:
# Define label mapping
label_names_3class = {
    0: "SEG_A",
    1: "SEG_B",
    2: "SEG_C"
}

In [6]:
# Create binary labels for SEG_A vs SEG_B/C
y_binary = tf.cast(y_labeled != 0, tf.int64)

binary_target_names = ["SEG_A", "SEG_BC"]

print(pd.Series(y_binary.numpy()).value_counts().sort_index())

0    6406
1    5493
Name: count, dtype: int64


In [7]:
# Fold assignment
test_fold = 3
val_fold = 4

train_mask = (fold_labeled != test_fold) & (fold_labeled != val_fold)
val_mask = fold_labeled == val_fold
test_mask = fold_labeled == test_fold

X_train = tf.boolean_mask(X_labeled, train_mask)
y_train = tf.boolean_mask(y_binary, train_mask)

X_val = tf.boolean_mask(X_labeled, val_mask)
y_val = tf.boolean_mask(y_binary, val_mask)

X_test = tf.boolean_mask(X_labeled, test_mask)
y_test = tf.boolean_mask(y_binary, test_mask)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (7140, 86, 65) (7140,)
Val: (2379, 86, 65) (2379,)
Test: (2380, 86, 65) (2380,)


In [ ]:
#Check distribution of classes in each split
def show_distribution(y_tensor, name):
    y_np = y_tensor.numpy()
    counts = pd.Series(y_np).value_counts().sort_index()
    pct = pd.Series(y_np).value_counts(normalize=True).sort_index() * 100
    
    dist = pd.DataFrame({
        "class": [binary_target_names[i] for i in counts.index],
        "count": counts.values,
        "pct": pct.values.round(2)
    })
    
    print(name)
    display(dist)

show_distribution(y_train, "Train")
show_distribution(y_val, "Validation")
show_distribution(y_test, "Test")

Train


,class,count,pct
0,SEG_A,3844,53.84
1,SEG_BC,3296,46.16


Validation


,class,count,pct
0,SEG_A,1281,53.85
1,SEG_BC,1098,46.15


Test


,class,count,pct
0,SEG_A,1281,53.82
1,SEG_BC,1099,46.18


In [9]:
# Compute class weights for binary classification
classes = np.unique(y_train.numpy())

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train.numpy()
)

class_weight_binary = dict(zip(classes, weights))
class_weight_binary

{np.int64(0): np.float64(0.9287200832466181),
 np.int64(1): np.float64(1.083131067961165)}

In [ ]:
# Get dimensions
n_weeks = X_train.shape[1]
n_features = X_train.shape[2]
n_classes = 2

print("n_weeks:", n_weeks)
print("n_features:", n_features)
print("n_classes:", n_classes)

In [ ]:
# %%
def build_binary_cnn(
    n_weeks,
    n_features,
    learning_rate=3e-4,
    dropout_config=(0.3, 0.3, 0.35, 0.4),
    l2_reg=1e-4,
    activation="relu"
):
    d1, d2, d3, d_dense = dropout_config
    
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    
    model = models.Sequential([
        layers.Input(shape=(n_weeks, n_features)),
        
        layers.BatchNormalization(),
        
        layers.Conv1D(
            filters=64,
            kernel_size=3,
            padding="same",
            activation=activation,
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(d1),
        
        layers.Conv1D(
            filters=64,
            kernel_size=5,
            padding="same",
            activation=activation,
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(d2),
        
        layers.Conv1D(
            filters=128,
            kernel_size=3,
            padding="same",
            activation=activation,
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.BatchNormalization(),
        layers.Dropout(d3),
        
        layers.GlobalAveragePooling1D(),
        
        layers.Dense(
            64,
            activation=activation,
            kernel_regularizer=regularizers.l2(l2_reg)
        ),
        layers.Dropout(d_dense),
        
        layers.Dense(2, activation="softmax")
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    return model